# Vacancies 2026 — Salary Prediction v7
**Key improvements over v6:**
- `mega_text = name_clean + key_skills_name + description` → TF-IDF(60k) + SVD(300)
- Smoothed TE (employer_id, employer_name, city, industries) inside CV folds
- LGB + XGB ensemble
- Ridge(alpha=1) on raw TF-IDF as 3rd model

**OOF benchmark (from experiments):**
- LGB + SVD(200) mega_text: **0.2138** vs v2 best Kaggle: 0.254

### 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

SEED     = 42
N_SPLITS = 5
STE_K    = 20
np.random.seed(SEED)

### 2. Data Loading

In [2]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test_x.csv')
print(f'train: {train.shape}, test: {test.shape}')

TARGET = 'salary_mean_net'
y_raw  = train[TARGET].values.astype(np.float64)
y_log  = np.log1p(y_raw).astype(np.float32)

train: (49051, 26), test: (12263, 25)


### 3. TF-IDF + SVD on mega_text (title + skills + description)

In [3]:
DESC_COL   = 'lemmaized_wo_stopwords_raw_description'
TITLE_COL  = 'name_clean'
SKILLS_COL = 'key_skills_name'

for df in [train, test]:
    df[DESC_COL]    = df[DESC_COL].fillna('')
    df[TITLE_COL]   = df[TITLE_COL].fillna('')
    df[SKILLS_COL]  = df[SKILLS_COL].fillna('')
    # mega_text: title + skills + description (all lemmatized)
    df['mega_text'] = df[TITLE_COL] + ' ' + df[SKILLS_COL] + ' ' + df[DESC_COL]

n_train = len(train)
corpus  = pd.concat([train['mega_text'], test['mega_text']], ignore_index=True)

print('Building TF-IDF (60k features)...')
tfidf = TfidfVectorizer(max_features=60000, ngram_range=(1, 2),
                        sublinear_tf=True, min_df=2, dtype=np.float32)
X_tfidf = tfidf.fit_transform(corpus)
print(f'TF-IDF shape: {X_tfidf.shape}')

print('Building SVD(300)...')
svd = TruncatedSVD(n_components=300, random_state=SEED)
X_svd_all = svd.fit_transform(X_tfidf).astype(np.float32)
X_svd_tr  = X_svd_all[:n_train]
X_svd_te  = X_svd_all[n_train:]
print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.3f}')
print(f'SVD train shape: {X_svd_tr.shape}')

Building TF-IDF (60k features)...
TF-IDF shape: (61314, 60000)
Building SVD(300)...
SVD explained variance: 0.236
SVD train shape: (49051, 300)


### 4. Tabular Feature Engineering

In [4]:
EXP_ORD = ['\u041d\u0435\u0442 \u043e\u043f\u044b\u0442\u0430',
           '\u041e\u0442 1 \u0433\u043e\u0434\u0430 \u0434\u043e 3 \u043b\u0435\u0442',
           '\u041e\u0442 3 \u0434\u043e 6 \u043b\u0435\u0442',
           '\u0411\u043e\u043b\u0435\u0435 6 \u043b\u0435\u0442']
exp_map = {v: i for i, v in enumerate(EXP_ORD)}

LOW_CARD = ['schedule_name', 'employment_name', 'specializations_profarea_name',
            'professional_roles_name', 'unified_address_region', 'unified_address_state',
            'if_foreign_language', 'is_branded_description']
TE_COLS  = ['employer_id', 'employer_name', 'unified_address_city', 'employer_industries']
NUM_COLS = ['experience_ord', 'has_languages', 'is_moscow', 'is_spb',
            'desc_len', 'title_len', 'skills_len', 'accept_handicapped', 'accept_kids']

for df in [train, test]:
    df['experience_ord'] = df['experience_name'].map(exp_map).fillna(-1).astype(np.int8)
    df['has_languages']  = (df['languages_name'].fillna('[]') != '[]').astype(np.int8)
    df['desc_len']       = df[DESC_COL].str.len().astype(np.int32)
    df['title_len']      = df[TITLE_COL].str.len().astype(np.int32)
    df['skills_len']     = df[SKILLS_COL].str.len().astype(np.int32)
    city = df['unified_address_city'].fillna('').str.lower()
    df['is_moscow']      = city.str.contains('\u043c\u043e\u0441\u043a\u0432\u0430').astype(np.int8)
    df['is_spb']         = city.str.contains('\u0441\u0430\u043d\u043a\u0442').astype(np.int8)
    df['accept_handicapped'] = df['accept_handicapped'].astype(np.int8)
    df['accept_kids']        = df['accept_kids'].astype(np.int8)
    for col in TE_COLS:
        if col in df.columns:
            df[col] = df[col].fillna('__NA__').astype(str)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.float32)
train[LOW_CARD] = oe.fit_transform(train[LOW_CARD].astype(str))
test[LOW_CARD]  = oe.transform(test[LOW_CARD].astype(str))

X_num_tr = train[NUM_COLS].values.astype(np.float32)
X_num_te = test[NUM_COLS].values.astype(np.float32)
X_low_tr = train[LOW_CARD].values.astype(np.float32)
X_low_te = test[LOW_CARD].values.astype(np.float32)

print(f'Num features: {X_num_tr.shape[1]}, Low-card: {X_low_tr.shape[1]}')
print(f'TE cols: {TE_COLS}')

Num features: 9, Low-card: 8
TE cols: ['employer_id', 'employer_name', 'unified_address_city', 'employer_industries']


### 5. CV Helpers (Smoothed TE inside folds)

In [6]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

def smoothed_te(col_tr, col_val, col_te, y_tr, gm, k=STE_K):
    """Smoothed target encoding: (n*mean + k*global) / (n+k)"""
    stats = pd.DataFrame({'y': y_tr, 'cat': col_tr.values}).groupby('cat')['y'].agg(['mean','count'])
    stats['smooth'] = (stats['count'] * stats['mean'] + k * gm) / (stats['count'] + k)
    te_map = stats['smooth']
    return (col_tr.map(te_map).fillna(gm).values,
            col_val.map(te_map).fillna(gm).values,
            col_te.map(te_map).fillna(gm).values)

def build_fold_arrays(tr_idx, val_idx):
    """Build (X_tr, X_val, X_te) numpy arrays for one fold."""
    gm = float(np.mean(y_log[tr_idx]))
    te_parts_tr, te_parts_val, te_parts_te = [], [], []

    for col in TE_COLS:
        s_tr, s_val, s_te = smoothed_te(
            train[col].iloc[tr_idx], train[col].iloc[val_idx], test[col], y_log[tr_idx], gm)
        te_parts_tr.append(s_tr); te_parts_val.append(s_val); te_parts_te.append(s_te)

    # Interaction: experience_ord × role TE
    role_tr, role_val, role_te = smoothed_te(
        train['professional_roles_name'].iloc[tr_idx],
        train['professional_roles_name'].iloc[val_idx],
        test['professional_roles_name'], y_log[tr_idx], gm)
    exp_tr  = train['experience_ord'].iloc[tr_idx].values.astype(float)
    exp_val = train['experience_ord'].iloc[val_idx].values.astype(float)
    exp_te  = test['experience_ord'].values.astype(float)
    te_parts_tr.append(exp_tr * role_tr)
    te_parts_val.append(exp_val * role_val)
    te_parts_te.append(exp_te * role_te)

    te_tr  = np.column_stack(te_parts_tr)
    te_val = np.column_stack(te_parts_val)
    te_te  = np.column_stack(te_parts_te)

    X_tr  = np.hstack([X_svd_tr[tr_idx],  X_low_tr[tr_idx],  X_num_tr[tr_idx],  te_tr])
    X_val = np.hstack([X_svd_tr[val_idx],  X_low_tr[val_idx], X_num_tr[val_idx], te_val])
    X_te  = np.hstack([X_svd_te,           X_low_te,          X_num_te,          te_te])
    return X_tr, X_val, X_te

print(f'Helpers ready. Total features per fold: {300 + len(LOW_CARD) + len(NUM_COLS) + len(TE_COLS) + 1}')

Helpers ready. Total features per fold: 322


### 6. LightGBM OOF

In [7]:
lgb_params = {
    'objective': 'regression', 'metric': 'rmse',
    'n_estimators': 8000, 'learning_rate': 0.02,
    'num_leaves': 255, 'min_child_samples': 15,
    'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.05, 'reg_lambda': 0.1,
    'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
}

oof_lgb  = np.zeros(len(train), dtype=np.float64)
pred_lgb = np.zeros(len(test),  dtype=np.float64)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_svd_tr)):
    X_tr, X_val, X_te = build_fold_arrays(tr_idx, val_idx)
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(X_tr, y_log[tr_idx], eval_set=[(X_val, y_log[val_idx])],
          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(1000)])
    vp = np.expm1(m.predict(X_val))
    oof_lgb[val_idx] = vp
    pred_lgb += np.expm1(m.predict(X_te)) / N_SPLITS
    mape = np.mean(np.abs((y_raw[val_idx] - vp) / (y_raw[val_idx] + 1e-8)))
    print(f'Fold {fold+1} | LGB MAPE: {mape:.4f} | iter: {m.best_iteration_}')

lgb_oof_mape = np.mean(np.abs((y_raw - oof_lgb) / (y_raw + 1e-8)))
print(f'\nLightGBM OOF MAPE: {lgb_oof_mape:.4f}')

[1000]	valid_0's rmse: 0.37395
Fold 1 | LGB MAPE: 0.2746 | iter: 1693
[1000]	valid_0's rmse: 0.366683
[2000]	valid_0's rmse: 0.36596
Fold 2 | LGB MAPE: 0.2667 | iter: 2359
[1000]	valid_0's rmse: 0.371687
[2000]	valid_0's rmse: 0.371138
Fold 3 | LGB MAPE: 0.2734 | iter: 1906
[1000]	valid_0's rmse: 0.363841
[2000]	valid_0's rmse: 0.363322
Fold 4 | LGB MAPE: 0.2677 | iter: 2018
[1000]	valid_0's rmse: 0.36487
[2000]	valid_0's rmse: 0.364409
Fold 5 | LGB MAPE: 0.2687 | iter: 1865

LightGBM OOF MAPE: 0.2702


### 7. XGBoost OOF

In [8]:
xgb_params = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse',
    'n_estimators': 8000, 'learning_rate': 0.02,
    'max_depth': 7, 'min_child_weight': 10,
    'subsample': 0.8, 'colsample_bytree': 0.6,
    'reg_alpha': 0.05, 'reg_lambda': 0.1,
    'random_state': SEED, 'n_jobs': -1, 'tree_method': 'hist',
    'early_stopping_rounds': 200, 'verbosity': 0,
}

oof_xgb  = np.zeros(len(train), dtype=np.float64)
pred_xgb = np.zeros(len(test),  dtype=np.float64)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_svd_tr)):
    X_tr, X_val, X_te = build_fold_arrays(tr_idx, val_idx)
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_tr, y_log[tr_idx], eval_set=[(X_val, y_log[val_idx])], verbose=False)
    vp = np.expm1(m.predict(X_val))
    oof_xgb[val_idx] = vp
    pred_xgb += np.expm1(m.predict(X_te)) / N_SPLITS
    mape = np.mean(np.abs((y_raw[val_idx] - vp) / (y_raw[val_idx] + 1e-8)))
    print(f'Fold {fold+1} | XGB MAPE: {mape:.4f} | iter: {m.best_iteration}')

xgb_oof_mape = np.mean(np.abs((y_raw - oof_xgb) / (y_raw + 1e-8)))
print(f'\nXGBoost OOF MAPE: {xgb_oof_mape:.4f}')

Fold 1 | XGB MAPE: 0.2699 | iter: 3752
Fold 2 | XGB MAPE: 0.2605 | iter: 5698
Fold 3 | XGB MAPE: 0.2685 | iter: 5861
Fold 4 | XGB MAPE: 0.2625 | iter: 4384
Fold 5 | XGB MAPE: 0.2623 | iter: 4013

XGBoost OOF MAPE: 0.2648


### 8. Ridge on raw TF-IDF (sparse)

In [9]:
# Ridge on raw TF-IDF (sparse) — complementary to tree models
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
cat_cols_ohe = ['experience_name', 'professional_roles_name', 'schedule_name',
                'employment_name', 'specializations_profarea_name', 'unified_address_region']
X_cat_tr = ohe.fit_transform(train[cat_cols_ohe].astype(str))
X_cat_te = ohe.transform(test[cat_cols_ohe].astype(str))

X_ridge_tr = sp.hstack([X_tfidf[:n_train], X_cat_tr])
X_ridge_te = sp.hstack([X_tfidf[n_train:], X_cat_te])
print(f'Ridge feature matrix: {X_ridge_tr.shape}')

oof_ridge  = np.zeros(len(train), dtype=np.float64)
pred_ridge = np.zeros(len(test),  dtype=np.float64)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_svd_tr)):
    m = Ridge(alpha=1.0)
    m.fit(X_ridge_tr[tr_idx], y_log[tr_idx])
    vp = np.expm1(m.predict(X_ridge_tr[val_idx]))
    oof_ridge[val_idx] = vp
    pred_ridge += np.expm1(m.predict(X_ridge_te)) / N_SPLITS
    mape = np.mean(np.abs((y_raw[val_idx] - vp) / (y_raw[val_idx] + 1e-8)))
    print(f'Fold {fold+1} | Ridge MAPE: {mape:.4f}')

ridge_oof_mape = np.mean(np.abs((y_raw - oof_ridge) / (y_raw + 1e-8)))
print(f'\nRidge OOF MAPE: {ridge_oof_mape:.4f}')

Ridge feature matrix: (49051, 60197)
Fold 1 | Ridge MAPE: 0.2426
Fold 2 | Ridge MAPE: 0.2428
Fold 3 | Ridge MAPE: 0.2447
Fold 4 | Ridge MAPE: 0.2442
Fold 5 | Ridge MAPE: 0.2429

Ridge OOF MAPE: 0.2434


### 9. Ensemble Blending

In [10]:
print(f'LGB   OOF MAPE: {np.mean(np.abs((y_raw - oof_lgb)   / (y_raw + 1e-8))):.4f}')
print(f'XGB   OOF MAPE: {np.mean(np.abs((y_raw - oof_xgb)   / (y_raw + 1e-8))):.4f}')
print(f'Ridge OOF MAPE: {np.mean(np.abs((y_raw - oof_ridge) / (y_raw + 1e-8))):.4f}')

# Grid search blend weights
best_mape = 1.0
best_w = (0.5, 0.4, 0.1)
print('\nBlend grid search (w_lgb, w_xgb, w_ridge):')
for wl in np.arange(0.3, 0.8, 0.1):
    for wx in np.arange(0.2, 0.7, 0.1):
        wr = round(1.0 - wl - wx, 2)
        if wr < 0 or wr > 0.4: continue
        blend = wl * oof_lgb + wx * oof_xgb + wr * oof_ridge
        mape = np.mean(np.abs((y_raw - blend) / (y_raw + 1e-8)))
        if mape < best_mape:
            best_mape = mape
            best_w = (round(wl,2), round(wx,2), wr)
            print(f'  w=({round(wl,2):.2f}, {round(wx,2):.2f}, {wr:.2f}): MAPE={mape:.4f}  <-- best')

print(f'\nBest blend: w_lgb={best_w[0]}, w_xgb={best_w[1]}, w_ridge={best_w[2]}')
print(f'Best blend OOF MAPE: {best_mape:.4f}')

wl, wx, wr = best_w
oof_final  = wl * oof_lgb  + wx * oof_xgb  + wr * oof_ridge
pred_final = wl * pred_lgb + wx * pred_xgb + wr * pred_ridge

LGB   OOF MAPE: 0.2702
XGB   OOF MAPE: 0.2648
Ridge OOF MAPE: 0.2434

Blend grid search (w_lgb, w_xgb, w_ridge):
  w=(0.30, 0.30, 0.40): MAPE=0.2381  <-- best

Best blend: w_lgb=0.3, w_xgb=0.3, w_ridge=0.4
Best blend OOF MAPE: 0.2381


### 10. Submission

In [11]:
sub = pd.DataFrame({'id': test['id'], 'salary_mean_net': pred_final})
sub.to_csv('submission_v7.csv', index=False)
print(f'Saved submission_v7.csv  shape={sub.shape}')
print(sub['salary_mean_net'].describe())
print(f'\nFinal OOF MAPE: {np.mean(np.abs((y_raw - oof_final) / (y_raw + 1e-8))):.4f}')

Saved submission_v7.csv  shape=(12263, 2)
count     12263.000000
mean      47724.305510
std       19060.140308
min       11481.416012
25%       35240.176589
50%       43696.602006
75%       55717.815607
max      163597.167161
Name: salary_mean_net, dtype: float64

Final OOF MAPE: 0.2381
